In [0]:
import pandas as pd
from pyspark.sql.functions import *

from pyspark.sql import functions as F

In [0]:
%sql
create catalog if not exists dev;

In [0]:

%sql
create database if not exists dev.ciencias_data

In [0]:
%sql
create volume if not exists dev.ciencias_data.session_data;

In [0]:
import pyspark.sql.functions as F
df=spark.read.format("csv").option("sep","|").option("header","true").load("/Volumes/dev/ciencias_data/session_data/sessions_part1.csv")
df=df.withColumn("_load_timestamp",F.lit(F.current_timestamp())).withColumn("_source",F.lit("Arkime"))
df.display()

In [0]:
from pyspark.sql.functions import to_timestamp, hour

df = df.withColumn(
    "event_timestamp",
    to_timestamp("timestamp")
).withColumn(
    "hour_partition",
    hour("event_timestamp")
)

df.write \
  .format("delta") \
  .mode("overwrite") \
  .partitionBy("hour_partition") \
  .saveAsTable("dev.ciencias_data.bronze_sessions")

In [0]:
df.display()

In [0]:
df_bronze = spark.table("dev.ciencias_data.bronze_sessions")

In [0]:
sample_json = df_bronze.select("data").limit(1).collect()[0][0]

In [0]:
from pyspark.sql.functions import schema_of_json

schema_str = spark.range(1).select(
    schema_of_json(F.lit(sample_json))
).collect()[0][0]

In [0]:
df_parsed = df_bronze.withColumn(
    "json_data",
    F.from_json(F.col("data"), schema_str)
).select("json_data.*", "event_timestamp", "_load_timestamp", "_source")

In [0]:
df_parsed.display()

In [0]:
df_parsed = df_parsed.drop("cert", "packetPos")

In [0]:
df_silver = df_parsed.withColumn(
    "total_packet_len",
    F.expr("""
        aggregate(packetLen, CAST(0 AS BIGINT),
                  (acc, x) -> acc + x)
    """)
)

In [0]:
df_silver = df_silver.withColumn(
    "avg_packet_len",
    F.col("total_packet_len") / F.size("packetLen")
).withColumn(
    "min_packet_len",
    F.array_min("packetLen")
).withColumn(
    "max_packet_len",
    F.array_max("packetLen")
).drop("packetLen")

df_silver.display()